# Pipeline INMET - Extração e Carga
Este notebook realiza o download, extração e conversão dos dados do INMET.

In [ ]:
!pip3 install -r requirements.txt -quiet

In [ ]:
import requests
import zipfile
import io
import os
import pandas as pd


In [ ]:
YEAR = "2026"

URL_DADOS = f"https://portal.inmet.gov.br/uploads/dadoshistoricos/{YEAR}.zip"

CAMINHO_BRONZE = f"/Users/mile/Documents/Projetos/pipeline-inmet-/data/bronze/inmet_clima/{YEAR}"

os.makedirs(CAMINHO_BRONZE, exist_ok=True)

print(f"Estrutura de pastas garantida em: {CAMINHO_BRONZE}")

Estrutura de pastas garantida em: /Users/mile/Documents/Projetos/pipeline-inmet-/data/bronze/inmet_clima/2026


In [ ]:
def download_data():
    print(f"Baixando os Dados do INMET para o ano: {YEAR}")

    response = requests.get(URL_DADOS, timeout=60)

    if response.status_code == 200:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            z.extractall(CAMINHO_BRONZE)

        print("Download concluído")
        return True

    else:
        print(f"Erro ao baixar os dados: {response.status_code}")
        return False


download_data()

Baixando os Dados do INMET para o ano: 2026
Download concluído


True

In [ ]:
!pip3 install pandas  -quiet

import pandas as pd

CAMINHO_BRONZE = f"/Users/mile/Documents/Projetos/pipeline-inmet-/data/bronze/inmet_clima/{YEAR}"

lista_arquivos = os.listdir(CAMINHO_BRONZE)

arquivos = [f for f in lista_arquivos if f.lower().endswith(".csv")]

dataframes = []

for arquivo in arquivos:

    caminho_csv = os.path.join(CAMINHO_BRONZE, arquivo)

    # leitura metadados
    with open(caminho_csv, "r", encoding="latin1") as f:
        cabecalho_linhas = [next(f).strip() for _ in range(8)]

    metadados = {}

    for linha in cabecalho_linhas:
        partes = linha.split(":", 1)

        if len(partes) == 2:
            metadados[partes[0].strip()] = partes[1].strip()

    # leitura da tabela
    df = pd.read_csv(
        caminho_csv,
        sep=";",
        encoding="latin1",
        skiprows=8
    )

    # adicionar metadados
    for chave, valor in metadados.items():
        df[chave] = valor

    dataframes.append(df)

# juntar todos os CSV
df_final = pd.concat(dataframes, ignore_index=True)

print(df_final.head())

zsh:1: /Users/mile/Documents/Projetos/pipeline-inmet-/venv/bin/pip3: bad interpreter: /Users/mile/Documents/Projetos/pipeline-inmet-/projeto/venv/bin/python3.14: no such file or directory

Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Execução do Pipeline
if download_data():
    # 3. Leitura Spark
    # Lê os arquivos CSV extraídos na pasta definida (CAMINHO_BRONZE)
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("sep", ";") \
        .option("encoding", "latin1") \
        .load(f"{CAMINHO_BRONZE}/*.csv")

    # 4. Adiciona coluna de partição
    df_final = df.withColumn("ano_particao", lit(int(YEAR)))

    # 5. Salva localmente
    df_final.write.format("parquet") \
        .mode("overwrite") \
        .partitionBy("ano_particao") \
        .save(OUTPUT_PATH)

    print(f"Pipeline local finalizada! Dados em: {OUTPUT_PATH}")
    
    # Exibe uma amostra dos dados
    df_final.show(5)